# DAN-DG pairwise source alignment
Written only; no cells executed during creation. See README.md for run order.

In [ ]:
"""DAN-DG: average Task 2 MMD across the three source-domain pairs."""
from itertools import combinations
from pathlib import Path
import math

import torch



def pairwise_source_mmd(features, batch_per_domain=8, scales=(0.5, 1.0, 2.0)):
    """Features are ordered Photo, Art Painting, Cartoon; bandwidths are per pair."""
    if features.ndim != 2 or features.shape != (3 * batch_per_domain, 512):
        raise ValueError('Expected three equal source blocks of 512-D features.')
    blocks = features.split(batch_per_domain)
    return sum(multi_kernel_mmd(a, b, scales) for a, b in combinations(blocks, 2)) / 3


def dan_dg_objective(model, x, y, cfg):
    features = forward_features(model, x)
    logits = model.fc(features)
    classification = nn.functional.cross_entropy(logits, y)
    alignment = pairwise_source_mmd(features, cfg['batch_per_domain'], cfg['kernel_scales'])
    return logits, classification + cfg['lambda_dg'] * alignment, classification, alignment
